In [0]:
# ------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime

In [0]:
# ------------------------------------------
# 2. ADF WIDGETS
# ------------------------------------------

dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("activity_name", "")
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("days_back", "30")
dbutils.widgets.text("environment", "dev")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
activity_name = dbutils.widgets.get("activity_name")
run_date = dbutils.widgets.get("run_date")
days_back = int(dbutils.widgets.get("days_back"))
environment = dbutils.widgets.get("environment")

print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Activity Name: {activity_name}")
print(f"Run Date: {run_date}")
print(f"Days Back: {days_back}")
print(f"Environment: {environment}")

Pipeline Run ID: 
Activity Name: 
Run Date: 
Days Back: 30
Environment: dev


In [0]:
# ------------------------------------------
# 3. CONNECT TO ADLS
# ------------------------------------------

storage_account_name = "staihospitaldev001"

storage_account_key = dbutils.secrets.get(
    scope="kv-aihoispital-scope",
    key="kv-aihospital-dev"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_path   = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

dq_log_path = gold_path + "data_quality_logs"

In [0]:
# ------------------------------------------
# 4. DQ LOGGING FUNCTION
# ------------------------------------------

def write_dq_log(check_name, table_name, status, failed_records=0, message=""):
    dq_df = spark.createDataFrame([
        Row(
            pipeline_run_id=pipeline_run_id,
            activity_name=activity_name,
            check_name=check_name,
            table_name=table_name,
            status=status,
            failed_records=int(failed_records),
            message=message,
            check_timestamp=datetime.utcnow()
        )
    ])

    dq_df.write.format("delta").mode("append").save(dq_log_path)

In [0]:
# ------------------------------------------
# 5. HELPER FUNCTIONS
# ------------------------------------------

def table_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def assert_table_not_empty(path, table_name):
    df = spark.read.format("delta").load(path)
    row_count = df.count()

    if row_count == 0:
        write_dq_log(
            check_name="Table Not Empty",
            table_name=table_name,
            status="Failed",
            failed_records=0,
            message=f"{table_name} has 0 rows"
        )
        raise Exception(f"Data quality failed: {table_name} has 0 rows")

    write_dq_log(
        check_name="Table Not Empty",
        table_name=table_name,
        status="Passed",
        failed_records=0,
        message=f"{table_name} has {row_count} rows"
    )

    return df


def assert_no_nulls(df, table_name, columns):
    for col_name in columns:
        null_count = df.filter(F.col(col_name).isNull()).count()

        if null_count > 0:
            write_dq_log(
                check_name="Null Check",
                table_name=table_name,
                status="Failed",
                failed_records=null_count,
                message=f"{table_name}.{col_name} has {null_count} null values"
            )
            raise Exception(f"Data quality failed: {table_name}.{col_name} has nulls")

        write_dq_log(
            check_name="Null Check",
            table_name=table_name,
            status="Passed",
            failed_records=0,
            message=f"{table_name}.{col_name} has no nulls"
        )


def assert_unique_key(df, table_name, key_column):
    duplicate_count = (
        df.groupBy(key_column)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    if duplicate_count > 0:
        write_dq_log(
            check_name="Duplicate Key Check",
            table_name=table_name,
            status="Failed",
            failed_records=duplicate_count,
            message=f"{table_name}.{key_column} has duplicate keys"
        )
        raise Exception(f"Data quality failed: {table_name}.{key_column} has duplicates")

    write_dq_log(
        check_name="Duplicate Key Check",
        table_name=table_name,
        status="Passed",
        failed_records=0,
        message=f"{table_name}.{key_column} is unique"
    )


def assert_probability_range(df, table_name, probability_column):
    invalid_count = (
        df.filter(
            (F.col(probability_column) < 0) |
            (F.col(probability_column) > 1) |
            (F.col(probability_column).isNull())
        )
        .count()
    )

    if invalid_count > 0:
        write_dq_log(
            check_name="Probability Range Check",
            table_name=table_name,
            status="Failed",
            failed_records=invalid_count,
            message=f"{table_name}.{probability_column} has invalid probability values"
        )
        raise Exception(f"Data quality failed: invalid probability values in {table_name}")

    write_dq_log(
        check_name="Probability Range Check",
        table_name=table_name,
        status="Passed",
        failed_records=0,
        message=f"{table_name}.{probability_column} values are between 0 and 1"
    )

In [0]:
# ------------------------------------------
# 6. START DATA QUALITY CHECKS
# ------------------------------------------

print("Starting data quality checks...")

# Bronze existence checks
bronze_tables = [
    "wards",
    "patients",
    "staff",
    "admissions",
    "incidents",
    "shift_assignments"
]

for table in bronze_tables:
    path = bronze_path + table

    if not table_exists(path):
        write_dq_log(
            check_name="Bronze Table Exists",
            table_name=table,
            status="Failed",
            failed_records=0,
            message=f"Bronze table missing: {table}"
        )
        raise Exception(f"Data quality failed: Bronze table missing: {table}")

    write_dq_log(
        check_name="Bronze Table Exists",
        table_name=table,
        status="Passed",
        failed_records=0,
        message=f"Bronze table exists: {table}"
    )

# Silver table checks
silver_admissions = assert_table_not_empty(
    silver_path + "admissions_enriched",
    "silver_admissions_enriched"
)

silver_incidents = assert_table_not_empty(
    silver_path + "incidents_enriched",
    "silver_incidents_enriched"
)

silver_staff = assert_table_not_empty(
    silver_path + "staff_activity",
    "silver_staff_activity"
)

assert_no_nulls(
    silver_admissions,
    "silver_admissions_enriched",
    ["AdmissionID", "PatientID", "WardName", "AdmissionDate"]
)

assert_unique_key(
    silver_admissions,
    "silver_admissions_enriched",
    "AdmissionID"
)

# Gold fact checks
gold_fact = assert_table_not_empty(
    gold_path + "patient_incident_features",
    "gold_patient_incident_features"
)

assert_no_nulls(
    gold_fact,
    "gold_patient_incident_features",
    ["AdmissionID", "PatientID", "WardName", "LengthOfStayDays", "WillIncidentOccur"]
)

assert_unique_key(
    gold_fact,
    "gold_patient_incident_features",
    "AdmissionID"
)

# Prediction checks
predictions = assert_table_not_empty(
    gold_path + "patient_incident_predictions",
    "patient_incident_predictions"
)

assert_no_nulls(
    predictions,
    "patient_incident_predictions",
    ["AdmissionID", "PatientID", "PredictedIncidentProbability"]
)

assert_unique_key(
    predictions,
    "patient_incident_predictions",
    "AdmissionID"
)

assert_probability_range(
    predictions,
    "patient_incident_predictions",
    "PredictedIncidentProbability"
)

print("All data quality checks passed successfully.")

Starting data quality checks...
All data quality checks passed successfully.


In [0]:
dq_logs = spark.read.format("delta").load(
    "abfss://gold@staihospitaldev001.dfs.core.windows.net/data_quality_logs"
)

display(dq_logs.orderBy(F.col("check_timestamp").desc()))

pipeline_run_id,activity_name,check_name,table_name,status,failed_records,message,check_timestamp
,,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-06T03:18:30.871225Z
,,Duplicate Key Check,patient_incident_predictions,Passed,0,patient_incident_predictions.AdmissionID is unique,2026-05-06T03:18:27.84608Z
,,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-06T03:18:25.646257Z
,,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PatientID has no nulls,2026-05-06T03:18:23.670107Z
,,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.AdmissionID has no nulls,2026-05-06T03:18:21.627009Z
,,Table Not Empty,patient_incident_predictions,Passed,0,patient_incident_predictions has 1147 rows,2026-05-06T03:18:19.775369Z
,,Duplicate Key Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.AdmissionID is unique,2026-05-06T03:18:17.252204Z
,,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.WillIncidentOccur has no nulls,2026-05-06T03:18:13.84323Z
,,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.LengthOfStayDays has no nulls,2026-05-06T03:18:12.141896Z
,,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.WardName has no nulls,2026-05-06T03:18:10.567303Z
